STEPS TO FOLLOW:


1.Goal:
Given a text sequence, predict the next word.
Example


Input  : "I am learning"
Output : "deep"

2.DATA COLLECTION

I am going to use the "quote_dataset.csv" dataset , It has --- number of quote.

3.TEXT PREPROCESSING

i.Lowercase text

ii.Remove punctuation

iii.Remove special characters

iv.Tokenization

4.CREATE VOCUBULARY

5.CREATE TRAINING SEQUENCE

6.APPLYING PADDING

7.CONVERT OUTPUT TO ONE HOT ENCODING

8.DEEP LEARNING MODEL()

9.TRAIN THE MODEL

10.PREDICT NEXT WORD

11.IMPROVE THE ACCURACY(EFFICIENCY)

12.CREATING UI

13.DEPLOY


In [1]:
# IMPORT ALL THE REQUIRED LIBRARIES
import numpy as np
import pandas as pd
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# IMPORT THE DATASET
df = pd.read_csv("qoute_dataset.csv")
#show the firt 5 entry
df.head()

In [ ]:
#NOW WE HAVE TO WORK ONLY ON THE QUOTE SO KEEP ONLY QUOTE.
quote = df['quote']
quote.head()

In [4]:
#STEP 3. TEXT PREPROCESSING
#1. LOWERCASE THE TEXT
quote = quote.apply(lambda x: x.lower())

In [5]:
import string
# 2 REMOVE THE PUNCTUATION
# create translation table
translator = str.maketrans('', '', string.punctuation)

# remove punctuation from the entire dataset column "quote"
quote = quote.apply(lambda x: x.translate(translator))

In [ ]:
#3 REMOVE SPECIAL CHARACTER (IF HAVE)
import re
quote = quote.apply(lambda x: re.sub(r'[^a-zA-Z0-9 ]', '', x))
df.head()

In [7]:
vocab_size = 10000

In [8]:
#4 TOKENIZATION  #5 VOCABULARY
from tensorflow.keras.preprocessing.text import Tokenizer

# create tokenizer
tokenizer = Tokenizer(num_words= vocab_size)

# fit tokenizer on the dataset
tokenizer.fit_on_texts(quote)

# convert sentences to token sequences
sequences = tokenizer.texts_to_sequences(quote)

#print(sequences)

In [ ]:
# TOTAL NUMBER OF TOKENS GENERATED
word_index = tokenizer.word_index
print(len(word_index))

In [ ]:
# PRINT THE FIRST QUOTE AND THEIR TOKENS
print(quote[0])
print(sequences[0])

In [11]:
# CREATING TRAINING SEQUENCE
# INPUT AND OUTPUT VARIABLE
X= []
y=[]
for seq in sequences:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)



In [ ]:
# PRINT THE LENGTH OF THE X
len(X)

In [ ]:
#PRINT THE LENGTH OF Y
len(y)

In [14]:
# CALCULATE THE MAX LENGTH FOR APPLYING THE PADDING
#6 APPLYING PADDING
max_len =max(len(x) for x in X)
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padding =  pad_sequences(X,maxlen = max_len,padding = 'pre')

In [15]:
y = np.array(y)

In [ ]:
#SHAPE OF PADDING
X_padding.shape

In [17]:
#7 CONVERT OUTPUT TO ONE HOT ENCODING
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes = vocab_size)


In [ ]:
y_one_hot.shape

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional

model = Sequential()
model.add(Embedding(vocab_size, 256, input_length=max_len))
model.add(Bidirectional(LSTM(256, return_sequences=True)))
model.add(LSTM(128))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

In [20]:
from tensorflow.keras.callbacks import EarlyStopping

# Define early stopping to stop training when loss stops decreasing
early_stop = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)

# Re-training with early stopping (example for 10 epochs)
# history = model.fit(X_padding, y_one_hot, epochs=10, callbacks=[early_stop], verbose=1)

In [ ]:
#9 TRAIN THE MODEL
history = model.fit(X_padding, y_one_hot, epochs=10, verbose=1, callbacks=[early_stop])

In [ ]:
#SAVE THE MODEL
model.save("word_pred_10_epochs.h5")
import pickle
pickle.dump(tokenizer, open("tokenizer_10_epochs.pkl", "wb"))



In [27]:
pickle.dump(max_len, open("max_len_10_epochs.pkl","wb"))

In [ ]:
import tensorflow as tf
import pickle

# Load the model
model = tf.keras.models.load_model('word_pred_10_epochs.h5')

# Load the tokenizer
with open('tokenizer_10_epochs.pkl', 'rb') as handle:
    tokenizer = pickle.load(handle)

# Load max_len
with open('max_len_10_epochs.pkl', 'rb') as handle:
    max_len = pickle.load(handle)

print('Model, tokenizer, and max_len loaded successfully!')

In [53]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
import string
import re

def predict_next_word(seed_text):
    # Preprocess input text to match training style
    text = seed_text.lower().translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)

    # Tokenize
    token_list = tokenizer.texts_to_sequences([text])[0]

    # Padding
    token_list = pad_sequences([token_list], maxlen=max_len, padding='pre')

    # Predict probabilities
    predicted_probs = model.predict(token_list, verbose=0)[0]
    predicted_word_index = np.argmax(predicted_probs)

    # Convert index back to word using tokenizer.index_word
    output_word = tokenizer.index_word.get(predicted_word_index, "")

    return output_word

# Get user input and predict
seed = input("Enter your text: ")
prediction = predict_next_word(seed)
print(f"Seed: '{seed}'")
print(f"Predicted next word: '{prediction}'")

Enter your text: this is a
Seed: 'this is a'
Predicted next word: 'man'
